In [ ]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

^C
--2026-03-18 12:31:59--  https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400395283 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.0-bin-hadoop3.tgz’

spark-3.5.0-bin-had 100%[===================>] 381.85M  7.25MB/s    in 48s     

2026-03-18 12:32:48 (7.89 MB/s) - ‘spark-3.5.0-bin-hadoop3.tgz’ saved [400395283/400395283]



In [47]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
sc = spark.sparkContext
spark

# Часть 1: Введение

## Создание Resilient Distributed Dataset

In [48]:
warandpeace = sc.textFile("warandsociety.txt")
warandpeace.count()

12851

In [42]:
nilFile = sc.textFile("nil")
nilFile.count()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/content/nil
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:208)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:291)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:287)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:291)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:287)
	at org.apache.spark.api.python.PythonRDD.getPartitions(P

In [43]:
warandpeace.take(10)

['Лев Николаевич Толстой',
 'Война и мир. Книга 1',
 '',
 'Война и мир – 1',
 '',
 ' ',
 ' http://www.lib.ru',
 '',
 'Аннотация ',
 '']

In [44]:
linesWithWar = warandpeace.filter(lambda x: "война" in x)
linesWithWar.first()

"– Еh bien, mon prince. Genes et Lucques ne sont plus que des apanages, des поместья, de la famille Buonaparte. Non, je vous previens, que si vous ne me dites pas, que nous avons la guerre, si vous vous permettez encore de pallier toutes les infamies, toutes les atrocites de cet Antichrist (ma parole, j'y crois) – je ne vous connais plus, vous n'etes plus mon ami, vous n'etes plus мой верный раб, comme vous dites. [Ну, что, князь, Генуа и Лукка стали не больше, как поместьями фамилии Бонапарте. Нет, я вас предупреждаю, если вы мне не скажете, что у нас война, если вы еще позволите себе защищать все гадости, все ужасы этого Антихриста (право, я верю, что он Антихрист) – я вас больше не знаю, вы уж не друг мой, вы уж не мой верный раб, как вы говорите.] Ну, здравствуйте, здравствуйте. Je vois que je vous fais peur, [Я вижу, что я вас пугаю,] садитесь и рассказывайте."

In [45]:
def time(f):
    import time
    t = time.process_time()
    f()
    print(f"Elapsed time: {int((time.process_time() - t)*1e9)} ns")
linesWithWar.unpersist()
linesWithWar.cache()
time(lambda: linesWithWar.count() )
time(lambda: linesWithWar.count() )

Elapsed time: 3432925 ns
Elapsed time: 3630284 ns


In [46]:
import re

wordCounts = warandpeace.flatMap(lambda line: re.findall(r'\w+', line.lower())).map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

!rm -rf warandpeace_histogram.txt

wordCounts.saveAsTextFile("warandpeace_histogram.txt")

!cat warandpeace_histogram.txt/part-* | head -n 20

('николаевич', 7)
('толстой', 16)
('война', 68)
('мир', 56)
('www', 2)
('lib', 2)
('льва', 3)
('лежит', 25)
('в', 11121)
('величественного', 8)
('здания', 9)
('литературы', 2)
('воссоздал', 2)
('россию', 35)
('этой', 315)
('на', 6799)
('сейчас', 156)
('щедростью', 2)
('души', 63)
('искренностью', 6)


## Операции с множествами

In [49]:
a = sc.parallelize([1,2,3,4])
b = sc.parallelize([3,4,6,7])

In [54]:
a.union(b).collect()

[1, 2, 3, 4, 3, 4, 6, 7]

In [53]:
a.union(b).distinct().collect()

[4, 1, 2, 6, 3, 7]

In [55]:
a.intersection(b).collect()

[4, 3]

In [56]:
a.subtract(b).collect()

[1, 2]

## Общие переменные

In [68]:
broadcastVar = sc.broadcast([1,2,3])

In [69]:
broadcastVar.value

[1, 2, 3]

In [70]:
accum = sc.accumulator(0)

In [71]:
sc.parallelize([1,2,3,4]).foreach(lambda x: accum.add(x))

In [72]:
accum.value

10

In [73]:
pair = ('a', 'b')

In [74]:
pair[0]

'a'

In [75]:
pair[1]

'b'

## Топ-10 популярных номеров такси

In [57]:
taxi = sc.textFile("nyctaxi.csv")

In [58]:
for t in taxi.take(5):
    print(t)

"_id","_rev","dropoff_datetime","dropoff_latitude","dropoff_longitude","hack_license","medallion","passenger_count","pickup_datetime","pickup_latitude","pickup_longitude","rate_code","store_and_fwd_flag","trip_distance","trip_time_in_secs","vendor_id"
"29b3f4a30dea6688d4c289c9672cb996","1-ddfdec8050c7ef4dc694eeeda6c4625e","2013-01-11 22:03:00",+4.07033460000000E+001,-7.40144200000000E+001,"A93D1F7F8998FFB75EEF477EB6077516","68BC16A99E915E44ADA7E639B4DD5F59",2,"2013-01-11 21:48:00",+4.06760670000000E+001,-7.39810790000000E+001,1,,+4.08000000000000E+000,900,"VTS"
"2a80cfaa425dcec0861e02ae44354500","1-b72234b58a7b0018a1ec5d2ea0797e32","2013-01-11 04:28:00",+4.08190960000000E+001,-7.39467470000000E+001,"64CE1B03FDE343BB8DFB512123A525A4","60150AA39B2F654ED6F0C3AF8174A48A",1,"2013-01-11 04:07:00",+4.07280540000000E+001,-7.40020370000000E+001,1,,+8.53000000000000E+000,1260,"VTS"
"29b3f4a30dea6688d4c289c96758d87e","1-387ec30eac5abda89d2abefdf947b2c1","2013-01-11 22:02:00",+4.07277180000000E+00

In [59]:
import itertools
taxi.mapPartitionsWithIndex(lambda idx, it:  itertools.islice(it,1,None) if (idx==0) else it  )

PythonRDD[124] at RDD at PythonRDD.scala:53

In [60]:
taxiParse = taxi.map(lambda line: line.split(","))

In [61]:
taxiMedKey = taxiParse.map(lambda row: (row[6], 1))

In [62]:
taxiMedCounts = taxiMedKey.reduceByKey(lambda v1, v2: v1+v2)

In [63]:
top10 = taxiMedCounts.map(lambda x: x[::-1]).top(10)
for x in top10:
    print(x[::-1])

('"AB44AD9A03B7CFAF3925103BDCC0AF23"', 44)
('"71CACFBADF9568AAE88A843DB511D172"', 41)
('"6483B9BFCB216EC88986EA3AB13064E7"', 41)
('"4C73459B430339981D78795300433438"', 41)
('"67E71D24AF704D814A0A825005ADA72E"', 40)
('"02E5A4136FD0A775A023A005A4EABC62"', 40)
('"9DFBCD218E7116F34C044F0680A0FB8A"', 39)
('"8DEB70907D00AA1D7FF5E2683240549B"', 39)
('"7989C2AB3F345F4AB54D3CF1E0480D67"', 39)
('"6C9F67DF658DC5636F9E7752F203F70A"', 39)


In [64]:
taxiCounts = taxi.map(lambda line: line.split(",")).map(lambda row: (row[6],1)).reduceByKey(lambda a,b: a + b)

In [65]:
taxiCounts.cache()

PythonRDD[134] at RDD at PythonRDD.scala:53

In [66]:
time(lambda: taxiCounts.count())
time(lambda: taxiCounts.count())

Elapsed time: 5219848 ns
Elapsed time: 3896025 ns


In [67]:
import pyspark
taxi.persist(storageLevel=pyspark.StorageLevel.MEMORY_ONLY)

nyctaxi.csv MapPartitionsRDD[122] at textFile at NativeMethodAccessorImpl.java:0

# Часть 2: Анализ данных велопарковок в интерактивном режиме

In [ ]:
from pyspark import SparkContext, SparkConf

conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster('local[*]')
sc = SparkContext(conf=conf)

In [76]:
tripData = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", 'M/d/y H:m') \
    .csv("trips.csv")

tripData

id,duration,start_date,start_station_name,start_station_id,end_date,end_station_name,end_station_id,bike_id,subscription_type,zip_code
4576,63,NULL,South Van Ness at...,66,2013-08-29 14:14:00,South Van Ness at...,66,520,Subscriber,94127
4607,NULL,2013-08-29 14:42:00,San Jose City Hall,10,2013-08-29 14:43:00,San Jose City Hall,10,661,Subscriber,95138
4130,71,2013-08-29 10:16:00,Mountain View Cit...,27,2013-08-29 10:17:00,Mountain View Cit...,27,48,Subscriber,97214
4251,77,2013-08-29 11:29:00,San Jose City Hall,10,2013-08-29 11:30:00,San Jose City Hall,10,26,Subscriber,95060
4299,83,2013-08-29 12:02:00,South Van Ness at...,66,2013-08-29 12:04:00,Market at 10th,67,319,Subscriber,94103
4927,103,2013-08-29 18:54:00,Golden Gate at Polk,59,2013-08-29 18:56:00,Golden Gate at Polk,59,527,Subscriber,94109
4500,109,2013-08-29 13:25:00,Santa Clara at Al...,4,2013-08-29 13:27:00,Adobe on Almaden,5,679,Subscriber,95112
4563,111,2013-08-29 14:02:00,San Salvador at 1st,8,2013-08-29 14:04:00,San Salvador at 1st,8,687,Subscriber,95112
4760,113,2013-08-29 17:01:00,South Van Ness at...,66,2013-08-29 17:03:00,South Van Ness at...,66,553,Subscriber,94103
4258,114,2013-08-29 11:33:00,San Jose City Hall,10,2013-08-29 11:35:00,MLK Library,11,107,Subscriber,95060


In [77]:
tripData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [78]:
tripData.show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|               NULL|South Van Ness at...|              66|2013-08-29 14:14:00|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|    NULL|2013-08-29 14:42:00|  San Jose City Hall|              10|2013-08-29 14:43:00|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|    

In [79]:
? tripData.dropna

In [80]:
tripData.dropna().show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [81]:
tripData.describe().show()

+-------+-----------------+------------------+--------------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+
|summary|               id|          duration|  start_station_name|  start_station_id|    end_station_name|    end_station_id|           bike_id|subscription_type|            zip_code|
+-------+-----------------+------------------+--------------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+
|  count|           669959|            669958|              669959|            669959|              669959|            669959|            669959|           669959|              663340|
|   mean|460382.0098991132| 1107.951395460611|                NULL| 57.85187601032302|                NULL|57.837437813358726|427.58761954089726|             NULL|  1576245.3147059095|
| stddev|264584.4584872604|22255.453593552545|                NULL|17.11247

In [82]:
stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv("stations.csv")

stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [83]:
stationData.show(n=5)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
only showing top 5 rows



In [84]:
stationData.describe().show()

+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|summary|                id|                name|                lat|               long|       dock_count|         city|
+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|  count|                70|                  70|                 70|                 70|               70|           70|
|   mean|              43.0|                NULL|  37.59024338428572|-122.21841616428571|17.65714285714286|         NULL|
| stddev|24.166091947189145|                NULL|0.20347253639672502|0.20944604979644524|4.010441857493954|         NULL|
|    min|                 2|       2nd at Folsom|          37.329732|        -122.418954|               11|Mountain View|
|    max|                84|Yerba Buena Cente...|           37.80477|        -121.877349|               27|     San Jose|
+-------+---------------

In [85]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [86]:
stationsView = stationData.select(stationData['id'], stationData['name'], stationData['lat'], stationData['long'])
stationsView.show()

+---+--------------------+------------------+-------------------+
| id|                name|               lat|               long|
+---+--------------------+------------------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|
|  8| San Salvador at 1st|         37.330165|-121.88583100000001|
|  9|           Japantown|         37.348742|-121.89471499999999|
| 10|  San Jose City Hall|         37.337391|        -121.886995|
| 11|         MLK Library|         37.335885|-121.88566000000002|
| 12|SJSU 4th at San C...|         37.332808|-121.88389099999999|
| 13|       St James Park|         37.339301|-121.88993700000002|
| 14|Arena

In [87]:
startTrips = tripData.select(tripData.id, tripData.duration, tripData.start_station_id).withColumnRenamed('id', 'trip_id').join(stationsView, tripData.start_station_id == stationsView.id)
startTrips = startTrips.drop('id')

In [89]:
startTrips.show()

+-------+--------+----------------+--------------------+------------------+-------------------+
|trip_id|duration|start_station_id|                name|               lat|               long|
+-------+--------+----------------+--------------------+------------------+-------------------+
|   4576|      63|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4607|    NULL|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|      71|              27|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|      77|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|      83|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4927|     103|              59| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|     109|               4|Santa Clara at Al...|         37.333988|        -121.894902|
|   4563|     111|               8| San 

In [90]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [91]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

In [92]:
endTrips = spark.sql("""
SELECT trips.id as trip_id, trips.end_station_id, trips.duration, stations.name as station_name, stations.lat, stations.long
FROM trips INNER JOIN stations
    ON trips.end_station_id==stations.id
""")

In [93]:
endTrips.show()

+-------+--------------+--------+--------------------+------------------+-------------------+
|trip_id|end_station_id|duration|        station_name|               lat|               long|
+-------+--------------+--------+--------------------+------------------+-------------------+
|   4576|            66|      63|South Van Ness at...|         37.774814|        -122.418954|
|   4607|            10|    NULL|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|            27|      71|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|            10|      77|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|            67|      83|      Market at 10th|37.776619000000004|-122.41738500000001|
|   4927|            59|     103| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|             5|     109|    Adobe on Almaden|         37.331415|          -121.8932|
|   4563|             8|     111| San Salvador at 1st|      

In [94]:
spark.sql("""
SELECT start_station_name, avg(duration)
FROM trips
GROUP BY trips.start_station_name
ORDER BY avg(duration) DESC
""").show()

+--------------------+------------------+
|  start_station_name|     avg(duration)|
+--------------------+------------------+
|University and Em...| 7090.239417989418|
|California Ave Ca...| 4628.005847953216|
|Redwood City Publ...| 4579.234741784037|
|       Park at Olive|4438.1613333333335|
|San Jose Civic Ce...| 4208.016938519448|
|Rengstorff Avenue...| 4174.082373782108|
|Redwood City Medi...| 3959.491961414791|
|Palo Alto Caltrai...|3210.6489815253435|
|San Mateo County ...|2716.7700348432054|
|    Broadway at Main|2481.2537313432836|
|Cowper at University|2477.2379912663755|
|Redwood City Calt...|2415.7175032175032|
|South Van Ness at...| 2401.603338509317|
|San Antonio Caltr...| 2377.497487437186|
|San Antonio Shopp...| 2285.981298129813|
|           Japantown| 2207.809947643979|
|Washington at Kea...|1979.3077445652175|
|Arena Green / SAP...|1963.9679144385027|
|SJSU 4th at San C...| 1962.280341880342|
|Castro Street and...|1868.3135135135135|
+--------------------+------------

In [105]:
! pip install h3 h3_pyspark pandas folium

  Using cached h3-4.4.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
Using cached h3-4.4.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.0 MB)


In [96]:
# year - the year to represent, from 1 to 9999
# month - the month-of-year to represent, from 1 (January) to 12 (December)
# day - the day-of-month to represent, from 1 to 31
# hour - the hour-of-day to represent, from 0 to 23
# min - the minute-of-hour to represent, from 0 to 59
# sec - the second-of-minute and its micro-fraction to represent, from 0 to 60. The value can be either an integer like 13 , or a fraction like 13.123. If the sec argument equals to 60, the seconds field is set to 0 and 1 minute is added to the final timestamp.
# timezone - the time zone identifier. For example, CET, UTC and etc.

spark.sql("""
SELECT bike_id, start_date, end_date
FROM trips
WHERE
    start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+
|bike_id|         start_date|           end_date|
+-------+-------------------+-------------------+
|    379|2014-12-25 22:10:00|2014-12-25 22:18:00|
|    709|2014-12-25 21:21:00|2014-12-25 21:27:00|
|    376|2014-12-25 20:40:00|2014-12-25 20:46:00|
|    541|2014-12-25 20:27:00|2014-12-25 20:32:00|
|    283|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    519|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|
|    495|2014-12-25 18:42:00|2014-12-25 18:44:00|
|    541|2014-12-25 18:28:00|2014-12-25 18:37:00|
|    585|2014-12-25 18:27:00|2014-12-25 18:37:00|
|    574|2014-12-25 18:12:00|2014-12-25 18:21:00|
|    630|2014-12-25 18:12:00|2014-12-25 18:22:00|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|
|    290|2014-12-25 18:01:00|2014-12-25 18:15:00|
|    451|2014-12-25 17:55:00|2014-12-25 18:04:00|
|    630|2014-12-25 17:55:00|2014-12-25 17:59:00|
|    574|2014-12-25 17:54:00|2014-12-25 17:59:00|


In [97]:
spark.sql("""
SELECT trips.bike_id, trips.start_date, trips.end_date, stations.name
FROM trips INNER JOIN stations
    ON trips.start_station_id == stations.id
WHERE
    bike_id == 583
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+--------------+
|bike_id|         start_date|           end_date|          name|
+-------+-------------------+-------------------+--------------+
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|Market at 10th|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|Market at 10th|
|    583|2014-12-25 12:14:00|2014-12-25 12:21:00| Market at 4th|
+-------+-------------------+-------------------+--------------+



In [106]:
from pyspark.sql import functions as F
import h3_pyspark
import h3
import pyspark.sql as sql

In [111]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def raw_geo_to_h3(lat, lng, res):
    if lat is None or lng is None:
        return None
    try:
        return h3.latlng_to_cell(float(lat), float(lng), int(res))
    except (ValueError, TypeError):
        return None
my_h3_udf = F.udf(raw_geo_to_h3, StringType())
spark.udf.register("geo_to_h3", raw_geo_to_h3, StringType())
resolution = 8
stationData_fixed = stationData.withColumn(
    'h3',
    my_h3_udf(F.col('lat'), F.col('long'), F.lit(resolution))
)
stationData_fixed.createOrReplaceTempView("stations_h3")

In [112]:
christmas_583_contacts = spark.sql("""
SELECT trips.bike_id, trips.start_date, stations_h3.h3, stations_h3.lat, stations_h3.long, stations_h3.name
FROM trips INNER JOIN stations_h3
    ON trips.start_station_id == stations_h3.id
WHERE
    stations_h3.h3 in (SELECT stations_h3.h3
                            FROM trips INNER JOIN stations_h3
                                ON trips.start_station_id == stations_h3.id
                            WHERE
                                bike_id == 583
                                AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
                                AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0))
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date < make_timestamp(2014, 12, 26, 0, 0, 0)
ORDER BY trips.start_date
""")
christmas_583_contacts.cache()
christmas_583_contacts.show()

+-------+-------------------+---------------+------------------+-------------------+------------------+
|bike_id|         start_date|             h3|               lat|               long|              name|
+-------+-------------------+---------------+------------------+-------------------+------------------+
|    439|2014-12-25 01:40:00|8828308281fffff|37.776619000000004|-122.41738500000001|    Market at 10th|
|    659|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    465|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    583|2014-12-25 12:14:00|88283082abfffff|         37.786305|-122.40496599999999|     Market at 4th|
|    479|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    331|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    330|2014-12-25 12:27:00|88283082abfffff|37.781752000000004|

In [117]:
import pandas as pd
import h3
import folium

h3_places = christmas_583_contacts.select('lat','long', 'name', 'h3').toPandas()

def init_map(hexagons, width=1100, height=900):
    lats = []
    longs = []
    for hexagon in hexagons:
        lat, lng = h3.cell_to_latlng(hexagon)
        lats.append(lat)
        longs.append(lng)
    avg_lat = sum(lats)/len(lats) if lats else 37.7749
    avg_lng = sum(longs)/len(longs) if longs else -122.4194
    return folium.Map(location=[avg_lat, avg_lng], zoom_start=14, tiles='cartodbpositron', width=width, height=height)

def visualize_hexagons(folium_map, hexagons, color="blue"):
    for hexagon in hexagons:
        boundary = h3.cell_to_boundary(hexagon)
        boundary_list = list(boundary)
        boundary_list.append(boundary_list[0])
        folium.PolyLine(
            locations=boundary_list,
            weight=5,
            color=color,
            opacity=0.8,
            fill=True,
            fill_color=color,
            fill_opacity=0.2
        ).add_to(folium_map)

    return folium_map

def visualize_stations(folium_map, stations):
    for row in stations.itertuples():
        lat, lng, station_name = row.lat, row.long, row.name
        folium.CircleMarker(
            location=(lat, lng),
            radius=5,
            color='red',
            fill=True,
            fill_color='red'
        ).add_to(folium_map)
        folium.map.Marker(
            location=(lat, lng),
            icon=folium.features.DivIcon(
                icon_size=(200,36),
                icon_anchor=(-15,35),
                html=f'<div style="font-size: 9pt; font-weight: bold; color: black; background: white; border: 1px solid black; padding: 2px; border-radius: 3px;">{station_name}</div>',
            )
        ).add_to(folium_map)
    return folium_map
if not h3_places.empty:
    unique_hexes = h3_places.h3.unique()
    m = init_map(unique_hexes)
    visualize_hexagons(m, unique_hexes, color="black")
    stations_unique = h3_places[['lat', 'long', 'name']].drop_duplicates()
    visualize_stations(m, stations_unique)
    display(m)
else:
    print("Ошибка: Таблица h3_places пуста. Проверьте результат SQL запроса.")

In [118]:
sc.stop()

In [119]:
from pyspark import SparkContext, SparkConf

In [120]:
conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster('local[*]')

In [121]:
sc = SparkContext(conf=conf)

In [122]:
tripData = sc.textFile("trips.csv")
# запомним заголовок, чтобы затем его исключить из данных
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("stations.csv")
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

In [123]:
list(enumerate(tripsHeader.split(",")))

[(0, 'id'),
 (1, 'duration'),
 (2, 'start_date'),
 (3, 'start_station_name'),
 (4, 'start_station_id'),
 (5, 'end_date'),
 (6, 'end_station_name'),
 (7, 'end_station_id'),
 (8, 'bike_id'),
 (9, 'subscription_type'),
 (10, 'zip_code')]

In [124]:
list(enumerate(stationsHeader.split(",")))

[(0, 'id'),
 (1, 'name'),
 (2, 'lat'),
 (3, 'long'),
 (4, 'dock_count'),
 (5, 'city'),
 (6, 'installation_date')]

In [125]:
trips.take(2)

[['4576',
  '63',
  '',
  'South Van Ness at Market',
  '66',
  '8/29/2013 14:14',
  'South Van Ness at Market',
  '66',
  '520',
  'Subscriber',
  '94127'],
 ['4607',
  '',
  '8/29/2013 14:42',
  'San Jose City Hall',
  '10',
  '8/29/2013 14:43',
  'San Jose City Hall',
  '10',
  '661',
  'Subscriber',
  '95138']]

In [126]:
stations.take(2)

[['2',
  'San Jose Diridon Caltrain Station',
  '37.329732',
  '-121.90178200000001',
  '27',
  'San Jose',
  '8/6/2013'],
 ['3',
  'San Jose Civic Center',
  '37.330698',
  '-121.888979',
  '15',
  'San Jose',
  '8/5/2013']]

In [127]:
stationsIndexed = stations.keyBy(lambda station: station[0])

In [128]:
stationsIndexed.take(2)

[('2',
  ['2',
   'San Jose Diridon Caltrain Station',
   '37.329732',
   '-121.90178200000001',
   '27',
   'San Jose',
   '8/6/2013']),
 ('3',
  ['3',
   'San Jose Civic Center',
   '37.330698',
   '-121.888979',
   '15',
   'San Jose',
   '8/5/2013'])]

In [129]:
tripsByStartTerminals = trips.keyBy(lambda trip: trip[4])
tripsByEndTerminals = trips.keyBy(lambda trip: trip[7])

In [130]:
startTrips = stationsIndexed.join(tripsByStartTerminals)
endTrips = stationsIndexed.join(tripsByEndTerminals)

In [131]:
print(startTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[23] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[15] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[14] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(5) PairwiseRDD[13] at join at /tmp/ipykernel_20497/210173577.py:1 []
    |  PythonRDD[12] at join at /tmp/ipykernel_20497/210173577.py:1 []
    |  UnionRDD[11] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[9] at RDD at PythonRDD.scala:53 []
    |  stations.csv MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv HadoopRDD[3] at textFile at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[10] at RDD at PythonRDD.scala:53 []
    |  trips.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  trips.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


In [132]:
print(endTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[24] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[22] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[21] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(5) PairwiseRDD[20] at join at /tmp/ipykernel_20497/210173577.py:2 []
    |  PythonRDD[19] at join at /tmp/ipykernel_20497/210173577.py:2 []
    |  UnionRDD[18] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[16] at RDD at PythonRDD.scala:53 []
    |  stations.csv MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv HadoopRDD[3] at textFile at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[17] at RDD at PythonRDD.scala:53 []
    |  trips.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  trips.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


In [133]:
startTrips.count()

669959

In [134]:
endTrips.count()

669959

In [135]:
from pyspark.rdd import portable_hash

stationsIndexed.partitionBy(numPartitions=trips.getNumPartitions(), partitionFunc=lambda x: portable_hash(x[0]))

MapPartitionsRDD[30] at mapPartitions at PythonRDD.scala:160

In [136]:
stationsIndexed.partitioner

In [137]:
from typing import NamedTuple
from datetime import datetime

In [138]:
def initStation(stations):
    class Station(NamedTuple):
        station_id: int
        name: str
        lat: float
        long: float
        dockcount: int
        landmark: str
        installation: str

    for station in stations:
        yield Station(
            station_id = int(station[0]),
            name = station[1],
            lat = float(station[2]),
            long = float(station[3]),
            dockcount = int(station[4]),
            landmark = station[5],
            installation = datetime.strptime(station[6], '%m/%d/%Y')
        )

In [139]:
stationsInternal = stations.mapPartitions(initStation)
stationsInternal.first()

Station(station_id=2, name='San Jose Diridon Caltrain Station', lat=37.329732, long=-121.90178200000001, dockcount=27, landmark='San Jose', installation=datetime.datetime(2013, 8, 6, 0, 0))

In [140]:
def initTrip(trips):
    class Trip(NamedTuple):
        trip_id: int
        duration: int
        start_date: datetime
        start_station_name: str
        start_station_id: int
        end_date: datetime
        end_station_name: str
        end_station_id: int
        bike_id: int
        subscription_type: str
        zip_code: str

    for trip in trips:
        try:
            yield Trip(
             trip_id = int(trip[0]),
             duration = int(trip[1]),
             start_date = datetime.strptime(trip[2], '%m/%d/%Y %H:%M'),
             start_station_name = trip[3],
             start_station_id = int(trip[4]),
             end_date = datetime.strptime(trip[5], '%m/%d/%Y %H:%M'),
             end_station_name = trip[6],
             end_station_id = trip[7],
             bike_id = int(trip[8]),
             subscription_type = trip[9],
             zip_code = trip[10]
            )
        except:
            pass

In [141]:
tripsInternal = trips.mapPartitions(initTrip)
tripsInternal.take(10)

[Trip(trip_id=4130, duration=71, start_date=datetime.datetime(2013, 8, 29, 10, 16), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 10, 17), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=48, subscription_type='Subscriber', zip_code='97214'),
 Trip(trip_id=4251, duration=77, start_date=datetime.datetime(2013, 8, 29, 11, 29), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 11, 30), end_station_name='San Jose City Hall', end_station_id='10', bike_id=26, subscription_type='Subscriber', zip_code='95060'),
 Trip(trip_id=4299, duration=83, start_date=datetime.datetime(2013, 8, 29, 12, 2), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 12, 4), end_station_name='Market at 10th', end_station_id='67', bike_id=319, subscription_type='Subscriber', zip_code='94103'),
 Trip(trip_id=4927, duration=103, s

In [142]:
tripsByStartStation = tripsInternal.keyBy(lambda trip: trip.start_station_name)

In [143]:
import numpy as np

avgDurationByStartStation = tripsByStartStation\
 .mapValues(lambda trip: trip.duration)\
 .groupByKey()\
 .mapValues(lambda trip_durations: np.mean(list(trip_durations)))

In [144]:
%%time

avgDurationByStartStation.top(10, key=lambda x: x[1])

CPU times: user 7.77 ms, sys: 608 µs, total: 8.38 ms
Wall time: 10.5 s


[('University and Emerson', np.float64(7090.239417989418)),
 ('California Ave Caltrain Station', np.float64(4628.005847953216)),
 ('Redwood City Public Library', np.float64(4579.234741784037)),
 ('Park at Olive', np.float64(4438.1613333333335)),
 ('San Jose Civic Center', np.float64(4208.016938519448)),
 ('Rengstorff Avenue / California Street', np.float64(4174.082373782108)),
 ('Redwood City Medical Center', np.float64(3959.491961414791)),
 ('Palo Alto Caltrain Station', np.float64(3210.6489815253435)),
 ('San Mateo County Center', np.float64(2716.7700348432054)),
 ('Broadway at Main', np.float64(2481.2537313432836))]

In [145]:
? tripsByStartStation.aggregateByKey

In [146]:
def seqFunc(acc, duration):
    duration_sum, count = acc
    return (duration_sum + duration, count + 1)

def combFunc(acc1, acc2):
    duration_sum1, count1 = acc1
    duration_sum2, count2 = acc2
    return (duration_sum1+duration_sum2, count1+count2)

def meanFunc(acc):
    duration_sum, count = acc
    return duration_sum/count

avgDurationByStartStation2 = tripsByStartStation\
  .mapValues(lambda trip: trip.duration)\
  .aggregateByKey(
    zeroValue=(0,0),
    seqFunc=seqFunc,
    combFunc=combFunc)\
  .mapValues(meanFunc)

In [147]:
%%time

avgDurationByStartStation2.top(10, key=lambda x: x[1])

CPU times: user 4.97 ms, sys: 2.08 ms, total: 7.05 ms
Wall time: 10.2 s


[('University and Emerson', 7090.239417989418),
 ('California Ave Caltrain Station', 4628.005847953216),
 ('Redwood City Public Library', 4579.234741784037),
 ('Park at Olive', 4438.1613333333335),
 ('San Jose Civic Center', 4208.016938519448),
 ('Rengstorff Avenue / California Street', 4174.082373782108),
 ('Redwood City Medical Center', 3959.491961414791),
 ('Palo Alto Caltrain Station', 3210.6489815253435),
 ('San Mateo County Center', 2716.7700348432054),
 ('Broadway at Main', 2481.2537313432836)]

In [148]:
def earliestTrip(trips):
    if trips is None:
        return None
    if len(trips)==0:
        return trips
    trips = list(trips)
    min_date = trips[0].start_date
    min_trip = trips[0]
    for trip in trips[1:]:
        if min_date > trip.start_date:
            min_date = trip.start_date
            min_trip = trip
    return min_trip

firstGrouped = tripsByStartStation\
  .groupByKey()\
  .mapValues(lambda trips: earliestTrip(trips))

In [149]:
%%time

firstGrouped.take(5)

CPU times: user 7.41 ms, sys: 1.24 ms, total: 8.65 ms
Wall time: 15.8 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id='5', bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id='10', bike_id=6

In [150]:
firstGrouped = tripsByStartStation\
  .reduceByKey(lambda tripA, tripB: tripA if tripA.start_date < tripB.start_date else tripB)

In [151]:
%%time

firstGrouped.take(5)

CPU times: user 7.26 ms, sys: 0 ns, total: 7.26 ms
Wall time: 9.58 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id='5', bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id='10', bike_id=6

In [152]:
sc.stop()

# Часть 3: Анализ данных велопарковок в неинтерактивном режиме

In [157]:
!rm -rf avg_duration_of_start_stations first_station_trips

In [158]:
!$SPARK_HOME/bin/spark-submit L1_noninteractive_bike_analysis_python.py

26/03/18 14:28:08 INFO SparkContext: Running Spark version 3.5.0
26/03/18 14:28:08 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/03/18 14:28:08 INFO SparkContext: Java version 11.0.30
26/03/18 14:28:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/18 14:28:08 INFO ResourceUtils: ==============================================================
26/03/18 14:28:08 INFO ResourceUtils: No custom resources configured for spark.driver.
26/03/18 14:28:08 INFO ResourceUtils: ==============================================================
26/03/18 14:28:08 INFO SparkContext: Submitted application: Lab1_Script
26/03/18 14:28:08 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus,

In [159]:
!head -n 5 avg_duration_of_start_stations/part-00000

('University and Emerson', 7090.239417989418)
('California Ave Caltrain Station', 4628.005847953216)
('Redwood City Public Library', 4579.234741784037)
('Park at Olive', 4438.1613333333335)
('San Jose Civic Center', 4208.016938519448)


In [160]:
!head -n 5 first_station_trips/part-00000

('Mountain View City Hall', Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id='27', bike_id=150, subscription_type='Subscriber', zip_code='97214'))
('Santa Clara at Almaden', Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id='5', bike_id=679, subscription_type='Subscriber', zip_code='95112'))
('San Salvador at 1st', Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id='10', bike_id=661, subscri

# Часть 4: Решение задач L1_Apache_Spark_Tasks.md

In [170]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import math

spark = SparkSession.builder.master("local[*]").getOrCreate()

trips = spark.read.csv("trips.csv", header=True, inferSchema=True)
stations = spark.read.csv("stations.csv", header=True, inferSchema=True)

bike_max = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total")) \
    .orderBy(F.desc("total")).first()
print(f"1. Велосипед с макс. пробегом: ID {bike_max['bike_id']}, время: {bike_max['total']} сек.")

def haversine(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]: return 0.0
    R = 6371
    dLat, dLon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dLat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dLon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
haversine_udf = F.udf(haversine, DoubleType())
st_a = stations.select(F.col("name").alias("station_a"),
                       F.col("lat").alias("lat_a"),
                       F.col("long").alias("long_a"))
st_b = stations.select(F.col("name").alias("station_b"),
                       F.col("lat").alias("lat_b"),
                       F.col("long").alias("long_b"))
max_dist_row = st_a.crossJoin(st_b) \
    .filter(F.col("station_a") < F.col("station_b")) \
    .withColumn("dist", haversine_udf("lat_a", "long_a", "lat_b", "long_b")) \
    .orderBy(F.desc("dist")).first()
print(f"2. Максимальное расстояние: {max_dist_row['dist']:.2f} км между '{max_dist_row['station_a']}' и '{max_dist_row['station_b']}'")

target_bike = bike_max['bike_id']
print(f"3. Путь велосипеда №{target_bike} через станции:")
trips.filter(F.col("bike_id") == target_bike) \
    .orderBy("start_date") \
    .select("start_date", "start_station_name", "end_station_name").show(truncate=False)

bike_count = trips.select("bike_id").distinct().count()
print(f"4. Всего уникальных велосипедов в системе: {bike_count}")

print("\n5. Пользователи с общим временем > 3 часов:")
trips.groupBy("zip_code") \
    .agg(F.sum("duration").alias("total_time")) \
    .filter((F.col("zip_code").isNotNull()) &
            (F.col("zip_code") != "nil") &
            (F.col("zip_code") != "")) \
    .filter("total_time > 10800") \
    .orderBy(F.desc("total_time")).show(10)

1. Велосипед с макс. пробегом: ID 535, время: 18611693 сек.
2. Максимальное расстояние: 69.92 км между 'Embarcadero at Sansome' и 'SJSU - San Salvador at 9th'
3. Путь велосипеда №535 через станции:
+---------------+---------------------------------------------+---------------------------------------------+
|start_date     |start_station_name                           |end_station_name                             |
+---------------+---------------------------------------------+---------------------------------------------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |
|1/1/2014 18:51 |Embarcadero at Sansome                       |Market at 4th                                |
|1/1/2014 19:48 |Market at 4th                                |South Van Ness at Market                     |
|1/10/2014 20:13|Market at 10th                               |Powell Street BART                           |
|1/10/2014 8:09 |Embarcadero at 